# Week 6: Data Visualization Mastery with Seaborn

## Interactive Sales Dashboard

This notebook explores the supplied 100-row sales dataset using Pandas, Seaborn, Matplotlib and Plotly.

## 1. Setup and imports
Install dependencies with `pip install -r requirements.txt` before running.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
DATA_PATH = 'sales_data.csv'  # keep the CSV beside this notebook

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Month'] = df['Date'].dt.to_period('M').astype(str)
display(df.head())
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())

## 3. Data validation and summary statistics
`Total_Sales` is the recorded transaction value. The validation below checks whether it equals `Quantity × Price`.

In [ ]:
df['Calculated_Sales'] = df['Quantity'] * df['Price']
df['Sales_Match'] = df['Total_Sales'].eq(df['Calculated_Sales'])
print('Rows with sales calculation match:', df['Sales_Match'].sum(), '/', len(df))
display(df[['Quantity','Price','Total_Sales','Calculated_Sales','Sales_Match']].head())
display(df.describe(include='all'))

## 4. Monthly sales trend

In [ ]:
monthly = df.groupby('Month', as_index=False)['Total_Sales'].sum()
sns.lineplot(data=monthly, x='Month', y='Total_Sales', marker='o')
plt.title('Monthly Sales Trend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Product and regional performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
product_sales = df.groupby('Product', as_index=False)['Total_Sales'].sum()
region_sales = df.groupby('Region', as_index=False)['Total_Sales'].sum()
sns.barplot(data=product_sales, x='Product', y='Total_Sales', hue='Product', palette='deep', legend=False, ax=axes[0])
axes[0].set_title('Revenue by Product')
sns.barplot(data=region_sales, x='Region', y='Total_Sales', hue='Region', palette='deep', legend=False, ax=axes[1])
axes[1].set_title('Revenue by Region')
plt.tight_layout()
plt.show()

## 6. Statistical visualizations: box plot and violin plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Product', y='Price', hue='Product', palette='deep', legend=False, ax=axes[0])
axes[0].set_title('Unit Price Distribution by Product')
sns.violinplot(data=df, x='Region', y='Total_Sales', hue='Region', palette='deep', inner='quartile', legend=False, ax=axes[1])
axes[1].set_title('Transaction Sales Distribution by Region')
plt.tight_layout()
plt.show()

## 7. Correlation heatmap

In [ ]:
corr = df[['Quantity', 'Price', 'Total_Sales']].corr()
sns.heatmap(corr, annot=True, cmap='vlag', center=0, fmt='.2f', square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Interactive Plotly charts

In [ ]:
fig = px.scatter(df, x='Quantity', y='Total_Sales', color='Product', size='Price',
                 hover_data=['Date', 'Customer_ID', 'Region'], title='Quantity vs Transaction Sales')
fig.show()
fig = px.sunburst(df, path=['Region', 'Product'], values='Total_Sales', title='Revenue by Region and Product')
fig.show()

## 9. Findings and interpretation
Use the printed aggregates below to report the strongest product and region. Avoid assuming causation from correlations.

In [ ]:
print(f"Total revenue: {df['Total_Sales'].sum():,.0f}")
print(f"Units sold: {df['Quantity'].sum():,}")
print(f"Unique customers: {df['Customer_ID'].nunique():,}")
print('Top product:', df.groupby('Product')['Total_Sales'].sum().idxmax())
print('Top region:', df.groupby('Region')['Total_Sales'].sum().idxmax())